# Arsitrad — Fine-Tune Gemma 4 2B for Indonesian Building Regulations
## RAG-Powered AI Chatbot for Indonesian Construction Law

**Author:** Hanif | UNDIP Architecture  
**Repo:** https://github.com/arsitekberotot/arsitrad

---

## Overview

This notebook fine-tunes **Gemma 4 2B E2B** with QLoRA on Indonesian building regulations using:
- **151 instruction-tuning examples** from `data/processed/arsitrad_training.jsonl`
- **ChromaDB RAG** for grounded generation
- **Unsloth** for 2x faster fine-tuning

**Runtime Requirements:**
| Section | GPU Required | Time |
|---------|-------------|------|
| Sections 1-3 (Setup + Data) | T4 (free) | ~10 min |
| Section 4 (Fine-tuning) | **A100** (Colab Pro) | ~30-60 min |

---

## Table of Contents
1. [Setup & Install](#1)
2. [Clone Repo & Load Training Data](#2)
3. [Test RAG Retrieval (No GPU)](#3)
4. [Fine-Tune Gemma 4 with QLoRA](#4)
5. [Test Fine-Tuned Model](#5)
6. [Push to HuggingFace (Optional)](#6)



In [ ]:
# Install git-lfs for large file support
!apt-get install -y git-lfs > /dev/null 2>&1
!git lfs install
print('git-lfs installed')

<a name='1'></a>
## 1. Setup & Installation


In [ ]:
# Install dependencies
!pip install -q --upgrade pip
!pip install -q \
    torch \
    transformers>=4.40.0 \
    peft>=0.10.0 \
    bitsandbytes \
    accelerate>=0.27.0 \
    scipy \
    chromadb>=0.4.0 \
    sentence-transformers>=2.5.0 \
    datasets>=2.16.0 \
    trl>=0.7.0 \
    gradio>=4.0.0 \
    pyyaml \
    tqdm \
    huggingface_hub

# Install Unsloth for 2x faster fine-tuning
!pip install -q unsloth

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected. Fine-tuning requires GPU.")


<a name='2'></a>
## 2. Clone Repo & Load Training Data


In [ ]:
# Clone your GitHub repo
!git clone https://github.com/arsitekberotot/arsitrad.git
%cd arsitrad

# Verify files
import os
print("Top-level contents:")
for f in os.listdir('.'):
    print(f"  {f}")
print()
print("Training data:")
for f in sorted(os.listdir('data/processed')):
    sz = os.path.getsize(f'data/processed/{f}')
    print(f"  {f}: {sz/1024:.1f} KB")


In [ ]:
# Load and inspect training dataset
import json
from collections import Counter

with open('data/processed/arsitrad_training.jsonl') as f:
    lines = f.readlines()

print(f"Total training examples: {len(lines)}")

cats = Counter()
for line in lines:
    ex = json.loads(line)
    cats[ex.get('category', 'unknown')] += 1

print("\nCategory distribution:")
for cat, count in cats.most_common():
    print(f"  {cat}: {count}")

ex = json.loads(lines[0])
print(f"\nExample (first 500 chars of 'text' field):")
print(ex['text'][:500])


<a name='3'></a>
## 3. Test RAG Retrieval (No GPU Required)


In [ ]:
# Setup ChromaDB + embedder
import chromadb
from sentence_transformers import SentenceTransformer
import numpy as np

CHROMA_DIR = 'data/processed/chroma_db'
client = chromadb.PersistentClient(path=CHROMA_DIR)
col = client.get_collection('arsitrad_national_regulations')
print(f"Collection: {col.name} | Chunks: {col.count()}")

print("Loading embedder...")
model_emb = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
print("Ready.")


In [ ]:
# Retrieval function with query normalization fix
def retrieve(query, top_k=5, min_similarity=0.3):
    raw_emb = model_emb.encode([query])[0]
    q_norm = raw_emb / np.linalg.norm(raw_emb)
    results = col.query(
        query_embeddings=[q_norm.tolist()],
        n_results=top_k,
        include=['documents', 'metadatas', 'embeddings']
    )
    outputs = []
    for doc, metadata, emb in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['embeddings'][0]
    ):
        if emb is None:
            continue
        stored = np.array(emb)
        cos_sim = float(np.dot(q_norm, stored) / (np.linalg.norm(stored) + 1e-8))
        if cos_sim >= min_similarity:
            outputs.append({'text': doc, 'source': metadata.get('source','?'), 'similarity': round(cos_sim, 4)})
    outputs.sort(key=lambda x: x['similarity'], reverse=True)
    return outputs

# Test
test_queries = [
    'Apa itu KDB dan KDH dalam bangunan gedung?',
    'Syarat izin mendirikan bangunan rumah tinggal',
    'Persyaratan struktur tahan gempa untuk gedung',
]

print("RAG Retrieval Test\n" + "="*50)
for q in test_queries:
    results = retrieve(q, top_k=3)
    print(f"\nQuery: {q}")
    print(f"  Retrieved: {len(results)} chunks")
    for r in results[:2]:
        print(f"    [{r['source']}] sim={r['similarity']}")
        print(f"    {r['text'][:100]}...")


<a name='4'></a>
## 4. Fine-Tune Gemma 4 2B with QLoRA

**GPU Required.** Free T4: reduce epochs to 1, batch_size to 2. A100 (Colab Pro): use 3 epochs, batch_size 4.


In [ ]:
# 4.1 Configuration
FINETUNE_CONFIG = {
    'model_name': 'google/gemma-4-E2B-it',
    'max_seq_length': 2048,
    'load_in_4bit': True,
    'lora_rank': 32,
    'lora_alpha': 64,
    'lora_dropout': 0.05,
    'batch_size': 4,
    'gradient_accumulation_steps': 4,
    'num_epochs': 3,
    'learning_rate': 2e-4,
    'warmup_steps': 20,
    'logging_steps': 10,
    'save_steps': 100,
    'output_dir': './arsitrad_finetuned',
    'use_gradient_checkpointing': True,
}

print("Fine-tune Configuration:")
for k, v in FINETUNE_CONFIG.items():
    print(f"  {k}: {v}")

effective = FINETUNE_CONFIG['batch_size'] * FINETUNE_CONFIG['gradient_accumulation_steps']
print(f"\nEffective batch size: {effective}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


In [ ]:
import unsloth
# 4.2 Load Gemma 4 + Apply LoRA
from unsloth import FastLanguageModel

print("Loading Gemma 4...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=FINETUNE_CONFIG['model_name'],
    max_seq_length=FINETUNE_CONFIG['max_seq_length'],
    load_in_4bit=FINETUNE_CONFIG['load_in_4bit'],
    dtype=None,
)
print("Gemma 4 loaded.")

print("Applying LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=FINETUNE_CONFIG['lora_rank'],
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                     'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=FINETUNE_CONFIG['lora_alpha'],
    lora_dropout=FINETUNE_CONFIG['lora_dropout'],
    bias='none',
    use_gradient_checkpointing=FINETUNE_CONFIG['use_gradient_checkpointing'],
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


In [ ]:
# 4.3 Load Dataset & Train
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

print("Loading training dataset...")
dataset = load_dataset('json', data_files='data/processed/arsitrad_training.jsonl', split='train')
print(f"Dataset: {len(dataset)} examples")

print("\nStarting QLoRA Fine-Tuning...")
print(f"  Epochs: {FINETUNE_CONFIG['num_epochs']}")
print(f"  Effective batch: {FINETUNE_CONFIG['batch_size'] * FINETUNE_CONFIG['gradient_accumulation_steps']}")
print(f"  LoRA rank: {FINETUNE_CONFIG['lora_rank']}")

training_args = TrainingArguments(
    output_dir=FINETUNE_CONFIG['output_dir'],
    num_train_epochs=FINETUNE_CONFIG['num_epochs'],
    per_device_train_batch_size=FINETUNE_CONFIG['batch_size'],
    gradient_accumulation_steps=FINETUNE_CONFIG['gradient_accumulation_steps'],
    learning_rate=FINETUNE_CONFIG['learning_rate'],
    weight_decay=0.01,
    warmup_steps=FINETUNE_CONFIG['warmup_steps'],
    logging_steps=FINETUNE_CONFIG['logging_steps'],
    save_steps=FINETUNE_CONFIG['save_steps'],
    save_total_limit=2,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_grad_norm=0.3,
    lr_scheduler_type='cosine',
    report_to='none',
    gradient_checkpointing=FINETUNE_CONFIG['use_gradient_checkpointing'],
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
    max_seq_length=FINETUNE_CONFIG['max_seq_length'],
    dataset_text_field='text',
)

trainer.train()
print("Training complete!")


In [ ]:
# 4.4 Save Fine-Tuned Model
print("Saving LoRA adapters...")
trainer.save_model(FINETUNE_CONFIG['output_dir'])
tokenizer.save_pretrained(FINETUNE_CONFIG['output_dir'])
print(f"Saved to: {FINETUNE_CONFIG['output_dir']}")

print("Saving merged model (for inference)...")
merged_path = FINETUNE_CONFIG['output_dir'] + '_merged'
model.save_pretrained_merged(merged_path, tokenizer)
print(f"Merged model: {merged_path}")

import os
model_size = sum(
    os.path.getsize(os.path.join(merged_path, f))
    for f in os.listdir(merged_path)
    if os.path.isfile(os.path.join(merged_path, f))
)
print(f"Model size: {model_size / 1024**2:.1f} MB")


<a name='5'></a>
## 5. Test Fine-Tuned Model


In [ ]:
# Quick inference test — reload merged model from disk
print("Loading merged model for inference...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=FINETUNE_CONFIG['output_dir'] + '_merged'
)
FastLanguageModel.for_inference(model)

SYSTEM = (
    "Kamu adalah Arsitrad, asisten AI untuk regulasi arsitektur Indonesia. "
    "Jawab dalam Bahasa Indonesia, cite sumber regulasi jika relevan."
)

test_prompt = (
    "<start_of_turn>system\n" + SYSTEM + "<end_of_turn>\n"
    "<start_of_turn>user\nApa syarat minimum ventilasi alami untuk rumah tinggal di Indonesia?<end_of_turn>\n"
    "<start_of_turn>model\n"
)

print("Generating...")
inputs = tokenizer([test_prompt], return_tensors='pt').to('cuda')
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.3,
    do_sample=True,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Inference Test:")
print("="*50)
print(response.split('<start_of_turn>model')[-1].strip()[:800])


In [ ]:
# Download model from Colab
from google.colab import files
import shutil

shutil.make_archive('/content/arsitrad_finetuned_merged', 'zip', merged_path)
print("Zipped: /content/arsitrad_finetuned_merged.zip")
files.download('/content/arsitrad_finetuned_merged.zip')
print("Download started.")


<a name='6'></a>
## 6. Push to HuggingFace Hub (Optional)


In [ ]:
# Push to HuggingFace
# Get HF token: https://huggingface.co/settings/tokens

from huggingface_hub import HfApi

HF_TOKEN = 'YOUR_HF_TOKEN_HERE'  # @param {type:"string"}
REPO_NAME = 'arsitrad-gemma4-indonesian-architecture'  # @param {type:"string"}

api = HfApi(token=HF_TOKEN)
repo_id = f'arsitekberotot/{REPO_NAME}'

api.create_repo(repo_id=repo_id, repo_type='model', exist_ok=True)
print(f"Repo: https://huggingface.co/{repo_id}")

api.upload_folder(
    folder_path=merged_path,
    repo_id=repo_id,
    repo_type='model',
)
print(f"Uploaded! Model available at: https://huggingface.co/{repo_id}")
